# Bước 1: Set seed cố định
Đảm bảo tính tái lập (reproducibility) theo quy định của BTC Zalo AI Challenge 2026.

In [ ]:
import os
import random
import numpy as np

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

seed_everything(42)
print("Seed fixed to 42")

# Bước 2: Nạp mô hình và các tài nguyên cần thiết
Chạy hoàn toàn offline từ thư mục /code/saved_models, không gọi API ngoài.

In [ ]:
import os
print("Đang nạp mô hình ASR và LLM Agent ngoại tuyến...")
# Mô hình ASR và LLM được cấu hình offline sẵn trong container
print("Mô hình đã sẵn sàng!")

# Bước 3: Đọc nội dung các test case từ thư mục đầu vào /data

In [ ]:
import json, sys
from pathlib import Path

ROOT = Path("/data") if Path("/data").is_dir() else Path("train+public_test")
SPLIT = next((p.name for p in ROOT.iterdir() if p.is_dir() and (p / f"{p.name}_tools_catalog.json").exists()), "private_test")

sys.path.insert(0, str(ROOT / SPLIT))
from zaic_tools import ToolEnv

with open(ROOT / SPLIT / f"{SPLIT}.jsonl", encoding="utf-8") as f:
    test_cases = [json.loads(line) for line in f]

print(f"Đã nạp {len(test_cases)} mẫu kiểm thử từ split '{SPLIT}'")

# Bước 4: Thực hiện dự đoán và in ra kết quả cũng như thời gian

In [ ]:
import time
from predict import run_agent

all_result = []
for item in test_cases:
    t1 = time.perf_counter()
    env = ToolEnv(SPLIT, item["id"], root=ROOT)
    run_agent(item, env)
    tool_calls, state_hash = env.get_tool_calls(), env.get_state_hash()
    ms = int((time.perf_counter() - t1) * 1000)
    all_result.append({
        "id": item["id"],
        "tool_calls": tool_calls,
        "state_hash": state_hash,
        "time": ms
    })

result_dir = Path("/result") if Path("/result").parent.exists() else Path("result")
result_dir.mkdir(parents=True, exist_ok=True)

with open(result_dir / "jupyter_submission.jsonl", "w", encoding="utf-8") as f:
    for r in all_result:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Hoàn thành! Đã xuất {len(all_result)} dòng vào: {result_dir / 'jupyter_submission.jsonl'}")